USE CASE: Sentiment_analysis

In [2]:
encoder_url="https://tfhub.dev/tensorflow/bert_en_uncased_L-12_H-768_A-12/4"
preprocess_url="https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3"

In [24]:
from transformers import BertTokenizer, BertModel
import torch
import torch.nn as nn


In [25]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")


In [ ]:
bert =BertModel.from_pretrained("bert-base-uncased")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7902.69it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [55]:
class SentimentModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = bert
        self.dropout = nn.Dropout(0.2) #on desactive aleatoirnement 20 pour cent des neurones pour eviter le overrfitting
        self.fc = nn.Linear(768, 2)  # 2 classes et 768 entree

    def forward(self, input_ids, attention_mask): #definir comment les data passe dans model
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        pooled = outputs.last_hidden_state[:, 0] #ca retourn un vecteur global qui represente tout la phrase
        x = self.dropout(pooled) # reguler
        return self.fc(x) #transformation vecteur en logits de deux valeur /classification

model = SentimentModel()

In [56]:
from datasets import load_dataset

dataset=load_dataset("imdb")

In [57]:
print(dataset)
print(dataset['train'][0])

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})
{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and

In [66]:
texts = dataset['train']['text'][:10]  # subset propre

inputs = tokenizer(
    texts,
    padding=True,
    truncation=True,
    return_tensors="pt"  # IMPORTANT
)

In [69]:
outputs = model(inputs["input_ids"], inputs["attention_mask"])
probs = torch.softmax(outputs, dim=1)

print(probs.tolist())

[[0.3598563075065613, 0.6401436924934387], [0.3559933006763458, 0.6440067887306213], [0.31232666969299316, 0.6876733303070068], [0.43751224875450134, 0.5624878406524658], [0.29504096508026123, 0.7049590349197388], [0.3517593443393707, 0.6482406258583069], [0.34991368651390076, 0.6500863432884216], [0.43610715866088867, 0.5638929009437561], [0.35566210746765137, 0.6443378925323486], [0.41752108931541443, 0.582478940486908]]
